In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Hello World: Ang Iyong Unang Quantum Circuit

Bumuo ng [Bell state](https://en.wikipedia.org/wiki/Bell_state) (dalawang magkakaugnay na qubit) at patakbuhin ito sa tatlong paraan:

1. **Ideal simulation** — perpektong mga resulta, walang kinakailangang account
2. **Noisy simulation** — tumutulad sa tunay na device, walang kinakailangang account
3. **Tunay na quantum hardware** — nangangailangan ng libreng IBM Quantum account (mga hakbang sa pag-setup sa ibaba)

## Bumuo ng circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## Opsyon 1: Ideal simulation (walang kinakailangang account)
Gumagamit ng `StatevectorSampler` — isang lokal na simulator na may perpekto at walang-ingay na mga resulta.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## Opsyon 2: Noisy simulation (walang kinakailangang account)
Gumagamit ng `FakeManilaV2` — isang lokal na simulator na tumutulad sa tunay na IBM quantum device, kasama ang mga katangian ng ingay nito. Dapat munang i-transpile (i-adjust) ang circuit sa gate set at qubit connectivity ng device.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Opsyon 3: Tunay na quantum hardware
Nangangailangan ng libreng IBM Quantum account. Para mag-set up ng isa:

1. Mag-register sa [quantum.cloud.ibm.com/registration](https://quantum.cloud.ibm.com/registration) — walang kinakailangang credit card para sa unang 30 araw
2. Mag-sign in sa [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) at piliin ang rehiyon na **us-east** (kinakailangan para sa libreng Open Plan)
3. Lumikha ng instance (libreng Open Plan) sa [Instances](https://quantum.cloud.ibm.com/instances) kung wala ka pa
4. Lumikha ng API key sa [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) (o sa [cloud.ibm.com/iam/apikeys](https://cloud.ibm.com/iam/apikeys))
5. Kopyahin ang iyong **CRN** (Cloud Resource Name) mula sa iyong pahina ng [Instances](https://quantum.cloud.ibm.com/instances)

Pagkatapos, kung hindi mo pa nase-save ang iyong mga credentials sa Binder session na ito, patakbuhin ang cell sa ibaba. Palitan ang `<your-api-key>` ng API key mula sa hakbang 4, at ang `<your-crn>` ng CRN mula sa hakbang 5.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="<your-api-key>",
    instance="<your-crn>",
    set_as_default=True,
    overwrite=True,
)

**Paalala:** Ang mga job sa tunay na hardware ay maaaring tumagal ng ilang oras depende sa queue times. Kung tumatakbo pa rin ang cell, maaari mong tingnan ang iyong job status at makita ang mga resulta sa [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Ano ang susunod?
- **[Tutorials](https://doqumentation.org/tutorials)** — sunud-sunod na mga gabay sa mga algorithm, error mitigation, transpilation, at iba pa
- **[Guides](https://doqumentation.org/guides)** — sangguniang materyal sa pagpapatakbo ng mga circuit, primitive, at ang Qiskit Runtime
- **[Courses](https://doqumentation.org/learning/courses/basics-of-quantum-information)** — nakaayos na mga landas ng pag-aaral mula sa mga pangunahing quantum hanggang sa utility-scale computing
- **[Modules](https://doqumentation.org/learning/modules/computer-science)** — malalim na mga konseptuwal na module sa computer science at quantum mechanics